# SHINE_SWE_OPENSOURCE_v2 Dataset Analysis

This notebook provides comprehensive analysis of the SHINE_SWE_OPENSOURCE_v2 dataset:
1. **Dataset Overview**: Number of sub-datasets, samples per dataset, data schema
2. **Repo Statistics**: Number of repos per dataset, trajectories per repo, correctness breakdown
3. **Issue Extraction Analysis**: Coverage of issue_content_hash, extraction methods
4. **Deduplication Analysis**: issue_content_hash uniqueness across datasets
5. **Token Length Distribution**: Using Qwen3.6-27B tokenizer with parallel processing
   - Per dataset / per repo / per role (system, user, assistant)
   - Summary tables with mean, max, min

**Key differences from v1:**
- v2 uses **Arrow format** (v1 used JSONL)
- v2 has **18 sub-datasets** (v1 had 10)
- 8 new datasets added: MEnvData-SWE, Nemotron-SFT-SWE-v3, Open-SWE-Traces, OpenSWE-Trajectory, SWE-Factory-Kimi-K2-2.8K, SWE-Factory-Kimi-K2-RS, Scale-SWE-Distilled-DeepSeek-v3.2, Scale-SWE-Distilled-DeepSeek-v4-Pro-High-41k
- CoderForge-Preview is **deduplicated** (413K → 258K by trajectory_id)
- All datasets have `issue_content_hash` field for cross-dataset deduplication

In [1]:
import json
import os
import sys
import time
import re
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
from multiprocessing import cpu_count
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.size'] = 11
matplotlib.rcParams['figure.figsize'] = (14, 6)
matplotlib.rcParams['figure.dpi'] = 100

DATA_DIR = Path('/apdcephfs_zwfy/share_303937731/xiyuanwang/liuyewei/SHINE_V2_tmp/data/SHINE_SWE_OPENSOURCE_v2')
MODEL_DIR = '../models/Qwen3.6-27B'

# Find all sub-dataset directories (Arrow IPC stream format)
# Each sub-dataset is a directory ending with .openai
dataset_dirs = sorted([d for d in DATA_DIR.iterdir() if d.is_dir() and d.name.endswith('.openai')])
print(f'Data directory: {DATA_DIR}')
print(f'Number of sub-datasets: {len(dataset_dirs)}')
print()
for d in dataset_dirs:
    arrow_files = list(d.glob('*.arrow'))
    total_size = sum(f.stat().st_size for f in arrow_files)
    size_str = f'{total_size/1e9:.2f} GB' if total_size > 1e9 else f'{total_size/1e6:.2f} MB'
    print(f'  {d.name:<75} {size_str}')

Data directory: /apdcephfs_zwfy/share_303937731/xiyuanwang/liuyewei/SHINE_V2_tmp/data/SHINE_SWE_OPENSOURCE_v2
Number of sub-datasets: 18

  CoderForge-Preview.openai                                                   55.39 GB
  Kwai-Klear-SWE-smith-mini-swe-agent-plus-trajectories-66k.openai            7.69 GB
  MEnvData-SWE-Trajectory.openai                                              832.55 MB
  Nemotron-SFT-SWE-v3.openai                                                  35.06 GB
  Nemotron-SWE-v1.openai                                                      10.89 GB
  Open-SWE-Traces.openai                                                      48.23 GB
  OpenSWE-Trajectory.openai                                                   3.18 GB
  SWE-Factory-Kimi-K2-2.8K.openai                                             235.91 MB
  SWE-Factory-Kimi-K2-RS.openai                                               61.91 MB
  SWE-Gym-OpenHands-SFT-Trajectories.openai                                   4

## 1. Load Dataset

The SHINE_SWE_OPENSOURCE_v2 dataset is stored in HuggingFace datasets Arrow format.
Each sub-dataset is a separate directory containing Arrow data files.

**Note**: Unlike v1 which used JSONL files, v2 uses Arrow IPC Stream format for much faster loading.

In [2]:
import pyarrow as pa
import pyarrow.ipc as ipc
import gc

# Metadata columns we need (lightweight, no messages content)
METADATA_COLS = ['source_dataset', 'repo', 'correctness', 'issue_content_hash',
                 'trajectory_id', 'instance_id', 'model']

def load_metadata_only(data_dir):
    """Load only lightweight metadata columns from all sub-datasets.
    
    This avoids loading the huge 'messages' column into Python memory.
    The full dataset is ~270GB total; messages alone account for >95% of that.
    By only loading metadata columns, we reduce memory from ~300GB to ~2GB.
    """
    dataset_dirs = sorted([d for d in data_dir.iterdir() if d.is_dir() and d.name.endswith('.openai')])
    all_metadata = []
    dataset_names_list = []
    
    for ds_dir in dataset_dirs:
        ds_name = ds_dir.name.replace('.openai', '')
        print(f'  Loading metadata: {ds_name}...', end=' ', flush=True)
        arrow_files = sorted(ds_dir.glob('*.arrow'))
        if not arrow_files:
            print('No arrow files, skipping.')
            continue
        try:
            ds_count = 0
            for arrow_file in arrow_files:
                reader = ipc.open_stream(str(arrow_file))
                table = reader.read_all()
                # Select only metadata columns that exist in this table
                available_cols = [c for c in METADATA_COLS if c in table.column_names]
                meta_table = table.select(available_cols)
                records = meta_table.to_pylist()
                # Add source_dataset if not in the table
                for rec in records:
                    if 'source_dataset' not in rec or not rec.get('source_dataset'):
                        rec['source_dataset'] = ds_name
                all_metadata.extend(records)
                ds_count += len(records)
                del table, meta_table, records
            dataset_names_list.append(ds_name)
            print(f'{ds_count:,} samples')
        except Exception as e:
            print(f'ERROR: {e}')
            continue
    
    gc.collect()
    print(f'\nTotal metadata records loaded: {len(all_metadata):,}')
    return all_metadata, dataset_names_list

print('Loading metadata (lightweight, no messages)...')
full_dataset, dataset_names_loaded = load_metadata_only(DATA_DIR)
print(f'\nFirst sample keys: {list(full_dataset[0].keys())}')

Loading metadata (lightweight, no messages)...
  Loading metadata: CoderForge-Preview... 258,134 samples
  Loading metadata: Kwai-Klear-SWE-smith-mini-swe-agent-plus-trajectories-66k... 65,994 samples
  Loading metadata: MEnvData-SWE-Trajectory... 3,918 samples
  Loading metadata: Nemotron-SFT-SWE-v3... 237,970 samples
  Loading metadata: Nemotron-SWE-v1... 51,029 samples
  Loading metadata: Open-SWE-Traces... 207,489 samples
  Loading metadata: OpenSWE-Trajectory... 12,431 samples
  Loading metadata: SWE-Factory-Kimi-K2-2.8K... 2,809 samples
  Loading metadata: SWE-Factory-Kimi-K2-RS... 729 samples
  Loading metadata: SWE-Gym-OpenHands-SFT-Trajectories... 491 samples
  Loading metadata: SWE-Hero-openhands-trajectories... 34,269 samples
  Loading metadata: SWE-Lego-Real-Data... 18,036 samples
  Loading metadata: SWE-Lego-Synthetic-Data... 11,471 samples
  Loading metadata: SWE-Zero-openhands-trajectories... 318,115 samples
  Loading metadata: SWE-rebench-openhands-trajectories... 67,07

In [3]:
# Inspect a sample to understand the data structure
# (Read one full record directly from Arrow since full_dataset only has metadata)
first_ds_dir = sorted([d for d in DATA_DIR.iterdir() if d.is_dir() and d.name.endswith('.openai')])[0]
first_arrow = sorted(first_ds_dir.glob('*.arrow'))[0]
reader = ipc.open_stream(str(first_arrow))
sample_table = reader.read_all().slice(0, 1)
sample = sample_table.to_pylist()[0]
del sample_table

print(f'=== Sample Structure (from {first_ds_dir.name}) ===')
for key, value in sample.items():
    if key == 'messages':
        msgs = value or []
        print(f'  {key}: list of {len(msgs)} messages')
        for i, msg in enumerate(msgs[:3]):
            role = msg.get('role', 'unknown')
            content = msg.get('content') or ''
            print(f'    [{i}] role={role}, content_len={len(content)}')
        if len(msgs) > 3:
            print(f'    ... ({len(msgs) - 3} more messages)')
    elif key == 'tools':
        if value:
            print(f'  {key}: list of {len(value)} tools')
        else:
            print(f'  {key}: {value}')
    elif isinstance(value, str) and len(value) > 100:
        print(f'  {key}: "{value[:100]}..." (len={len(value)})')
    else:
        print(f'  {key}: {value}')

=== Sample Structure (from CoderForge-Preview.openai) ===
  messages: list of 157 messages
    [0] role=system, content_len=5715
    [1] role=user, content_len=4769
    [2] role=assistant, content_len=188
    ... (154 more messages)
  source_dataset: CoderForge-Preview
  repo: qingyangwu/aiohttp_final
  correctness: incorrect
  instance_id: None
  trajectory_id: qingyangwu_aiohttp_final_006fbe03fede4eaa1eeba7b8393cbf4d63cb44b6_run1
  traj_id: None
  model: None
  resolved: None
  license: Apache-2.0
  tools: list of 4 tools
  issue_title: DNS Resolution Tasks Not Cancelled When Closing TCPConnector
  issue_content: "**Title:** DNS Resolution Tasks Not Cancelled When Closing TCPConnector

**Description:**
When the `..." (len=1078)
  issue_extraction_method: issue_description_tag
  issue_content_hash: c311059a06d85c78d8632f32


## 2. Load Tokenizer (Qwen3.6-27B)

In [4]:
from transformers import AutoTokenizer

print(f'Loading tokenizer from: {MODEL_DIR}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)
print(f'Tokenizer loaded: vocab_size={tokenizer.vocab_size}')
print(f'Tokenizer type: {type(tokenizer).__name__}')

/opt/conda/envs/torch-base/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/conda/envs/torch-base/lib/python3.13/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Loading tokenizer from: ../models/Qwen3.6-27B
Tokenizer loaded: vocab_size=248044
Tokenizer type: TokenizersBackend


## 3. Parallel Tokenization & Statistics Computation

We tokenize each message in parallel for efficiency.
Each sample's metadata (source_dataset, repo, correctness) and per-message token lengths are extracted.

**Note**: Since v2 uses Arrow format (much faster to load than JSONL), we can process all data in-memory.

In [5]:
import pickle
import multiprocessing as mp

NUM_TOKENIZE_WORKERS = min(64, cpu_count())
CHECKPOINT_DIR = Path('.') / 'cache' / 'checkpoints_opensource_v2'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_FILE = CHECKPOINT_DIR / 'all_results.pkl'

print(f'Tokenize workers: {NUM_TOKENIZE_WORKERS}')
print(f'Checkpoint directory: {CHECKPOINT_DIR.resolve()}')
print(f'Available CPU cores: {cpu_count()}')

def extract_repo_from_metadata(d):
    """Extract repo name from various metadata fields when 'repo' is null.
    
    Strategies:
    1. CoderForge-Preview: trajectory_id format '{prefix}_{repo}_final_{hash}_run{N}'
    2. SWE-smith / Kwai-Klear: instance_id format '{owner}__{repo}.{hash}.{suffix}'
    3. Others: try 'repo' field directly
    """
    # Try direct 'repo' field
    repo = d.get('repo')
    if repo and repo != 'unknown' and repo.strip():
        return repo
    
    # Try trajectory_id (CoderForge-Preview)
    trajectory_id = d.get('trajectory_id')
    if trajectory_id:
        m = re.match(r'^[^_]+_(.+?)_final_[0-9a-f]+_run\d+$', trajectory_id)
        if m:
            return m.group(1)
    
    # Try instance_id (SWE-smith, Kwai-Klear)
    instance_id = d.get('instance_id')
    if instance_id:
        m = re.match(r'^([^_]+)__([^.]+)\.', instance_id)
        if m:
            return f"{m.group(1)}/{m.group(2)}"
    
    return 'unknown'

# --- Multiprocessing-based tokenization (bypasses GIL) ---

def _init_tokenize_worker(model_dir: str):
    """Pool initializer: load tokenizer once per worker process."""
    global _worker_tokenizer
    from transformers import AutoTokenizer
    _worker_tokenizer = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True)

def _fix_messages_for_tokenize(messages):
    """Fix messages loaded from arrow files for apply_chat_template."""
    import json as _json
    fixed = []
    for msg in messages:
        msg = dict(msg)
        for key in ('name', 'tool_call_id'):
            if key in msg and msg[key] == '':
                del msg[key]
        if 'reasoning_content' in msg and msg['reasoning_content'] == '':
            del msg['reasoning_content']
        if msg.get('role') == 'assistant' and msg.get('content') == '' and msg.get('tool_calls'):
            del msg['content']
        if 'tool_calls' in msg:
            if not msg['tool_calls']:
                del msg['tool_calls']
            else:
                fixed_tc = []
                for tc in msg['tool_calls']:
                    tc = dict(tc)
                    if 'function' in tc and tc['function']:
                        func = dict(tc['function'])
                        if 'arguments' in func and isinstance(func['arguments'], str):
                            try:
                                func['arguments'] = _json.loads(func['arguments'])
                            except (ValueError, TypeError):
                                pass
                        tc['function'] = func
                    fixed_tc.append(tc)
                msg['tool_calls'] = fixed_tc
        fixed.append(msg)
    return fixed

def _fix_tools_for_tokenize(tools):
    """Fix tool definitions loaded from arrow files."""
    import json as _json
    if not tools:
        return None
    fixed = []
    for tool in tools:
        tool = dict(tool)
        if 'function' in tool and tool['function']:
            func = dict(tool['function'])
            if 'parameters' in func and func['parameters']:
                params = dict(func['parameters'])
                if 'properties' in params and isinstance(params['properties'], str):
                    try:
                        params['properties'] = _json.loads(params['properties'])
                    except (ValueError, TypeError):
                        pass
                func['parameters'] = params
            tool['function'] = func
        fixed.append(tool)
    return fixed

def _tokenize_one_sample(args):
    """Worker function: tokenize each message content with tokenizer.encode.
    
    We only need per-message token counts for visualization.
    apply_chat_template is NOT needed here (it's for training only).
    This is ~10x faster than apply_chat_template on long trajectories.
    """
    idx, messages, tools, metadata = args
    try:
        global _worker_tokenizer
        
        # Compute per-message token counts
        msg_token_info = []
        total_tokens = 0
        for msg in (messages or []):
            role = msg.get('role', 'unknown') if msg else 'unknown'
            content = (msg.get('content') or '') if msg else ''
            token_count = len(_worker_tokenizer.encode(content))
            msg_token_info.append((role, token_count))
            total_tokens += token_count
        
        repo = metadata.get('repo', 'unknown')
        if not repo or repo == 'unknown':
            repo = extract_repo_from_metadata(metadata)
        
        return {
            'source_dataset': metadata.get('source_dataset', 'unknown'),
            'repo': repo,
            'correctness': metadata.get('correctness', 'unknown'),
            'model': metadata.get('model', 'unknown'),
            'msg_token_info': msg_token_info,
            'total_tokens': total_tokens,
        }
    except Exception as e:
        return None

Tokenize workers: 64
Checkpoint directory: /apdcephfs_zwfy/share_303937731/xiyuanwang/liuyewei/SHINE_V2_tmp/data_visualize/cache/checkpoints_opensource_v2
Available CPU cores: 384


In [ ]:
def process_dataset_multiprocess(data_dir, metadata_list, model_dir, num_workers=NUM_TOKENIZE_WORKERS):
    """Stream-process Arrow files with true multiprocessing tokenization.
    
    Uses multiprocessing.Pool to bypass GIL and achieve real parallelism
    for CPU-intensive tokenization (apply_chat_template).
    Supports checkpoint/resume.
    """
    
    # Check for existing checkpoint
    if CHECKPOINT_FILE.exists():
        print(f'Loading checkpoint from {CHECKPOINT_FILE}...')
        with open(CHECKPOINT_FILE, 'rb') as f:
            results = pickle.load(f)
        expected_total = len(metadata_list)
        if len(results) >= expected_total:
            print(f'Checkpoint complete: {len(results):,} samples. Skipping processing.')
            return results[:expected_total]
        else:
            print(f'Checkpoint partial: {len(results):,}/{expected_total:,} samples. Resuming...')
            start_idx = len(results)
    else:
        results = []
        start_idx = 0
    
    dataset_dirs = sorted([d for d in data_dir.iterdir() if d.is_dir() and d.name.endswith('.openai')])
    
    total_expected = len(metadata_list)
    global_idx = 0
    start_time = time.time()
    last_report_time = start_time
    num_failed = 0
    
    print(f'Multiprocess tokenization from Arrow files ({num_workers} workers)...')
    print(f'Total samples: {total_expected:,}, starting from idx {start_idx:,}')
    
    # Create process pool with tokenizer loaded in each worker
    pool = mp.Pool(
        processes=num_workers,
        initializer=_init_tokenize_worker,
        initargs=(str(model_dir),),
    )
    
    try:
        for ds_dir in dataset_dirs:
            ds_name = ds_dir.name.replace('.openai', '')
            arrow_files = sorted(ds_dir.glob('*.arrow'))
            if not arrow_files:
                continue
            
            for arrow_file in arrow_files:
                try:
                    reader = ipc.open_stream(str(arrow_file))
                except Exception as e:
                    print(f'  ERROR opening {arrow_file.name}: {e}')
                    continue
                
                while True:
                    try:
                        batch = reader.read_next_batch()
                    except StopIteration:
                        break
                    
                    batch_rows = batch.num_rows
                    batch_global_start = global_idx
                    batch_global_end = global_idx + batch_rows
                    
                    # Skip batches entirely before resume point
                    if batch_global_end <= start_idx:
                        global_idx += batch_rows
                        del batch
                        continue
                    
                    # Extract messages from this batch (skip tools - not needed for encode-only)
                    col_names = batch.schema.names
                    messages_list = batch.column('messages').to_pylist() if 'messages' in col_names else [[] for _ in range(batch_rows)]
                    
                    # Determine which rows in this batch to process
                    local_start = max(0, start_idx - batch_global_start)
                    
                    # Build work items for the pool
                    work_items = []
                    for i in range(local_start, batch_rows):
                        gi = batch_global_start + i
                        meta = metadata_list[gi] if gi < len(metadata_list) else {}
                        work_items.append((gi, messages_list[i], None, meta))
                    
                    # Dispatch to pool with imap_unordered for best throughput
                    for result in pool.imap_unordered(_tokenize_one_sample, work_items, chunksize=128):
                        if result is not None:
                            results.append(result)
                        else:
                            num_failed += 1
                    
                    global_idx += batch_rows
                    del batch, messages_list, work_items
                    
                    # Progress report every 30 seconds
                    now = time.time()
                    if now - last_report_time > 30:
                        last_report_time = now
                        elapsed = now - start_time
                        processed = len(results) - (start_idx if start_idx > 0 else 0)
                        rate = processed / elapsed if elapsed > 0 else 0
                        remaining_samples = total_expected - len(results)
                        eta = remaining_samples / rate if rate > 0 else 0
                        pct = len(results) / total_expected * 100
                        print(f'  {pct:.1f}% ({len(results):,}/{total_expected:,}) | '
                              f'{ds_name[:30]} | '
                              f'rate={rate:.0f}/s | ETA={eta/60:.1f}min | failed={num_failed}', flush=True)
                
                gc.collect()
                
                # Print per-file completion
                elapsed = time.time() - start_time
                processed = len(results) - (start_idx if start_idx > 0 else 0)
                rate = processed / elapsed if elapsed > 0 else 0
                remaining_samples = total_expected - len(results)
                eta = remaining_samples / rate if rate > 0 else 0
                pct = len(results) / total_expected * 100
                print(f'  {pct:.1f}% ({len(results):,}/{total_expected:,}) | '
                      f'done: {ds_name[:30]}/{arrow_file.name} | '
                      f'rate={rate:.0f}/s | ETA={eta/60:.1f}min', flush=True)
                
                # Save checkpoint after each file
                with open(CHECKPOINT_FILE, 'wb') as f:
                    pickle.dump(results, f)
    finally:
        pool.close()
        pool.join()
    
    # Final checkpoint save
    with open(CHECKPOINT_FILE, 'wb') as f:
        pickle.dump(results, f)
    
    total_time = time.time() - start_time
    print(f'\nDone! Processed {len(results):,} samples in {total_time/60:.1f} minutes (failed={num_failed})')
    return results

all_data = process_dataset_multiprocess(DATA_DIR, full_dataset, MODEL_DIR)

## 4. Repo Statistics

For each dataset:
- How many unique repos
- For each repo: how many trajectories, how many correct / incorrect / unknown

In [ ]:
# Build per-dataset, per-repo statistics
dataset_repo_stats = defaultdict(lambda: defaultdict(lambda: Counter()))

for sample in all_data:
    ds = sample['source_dataset']
    repo = sample['repo']
    correctness = sample['correctness']
    dataset_repo_stats[ds][repo][correctness] += 1

# Summary table: per dataset
print('=' * 120)
print(f'{"Dataset":<55} {"#Repos":>7} {"#Trajectories":>14} {"#Correct":>9} {"#Incorrect":>11} {"#Unknown":>9}')
print('=' * 120)

dataset_summary_rows = []
for ds in sorted(dataset_repo_stats.keys()):
    repos = dataset_repo_stats[ds]
    n_repos = len(repos)
    total = sum(sum(c.values()) for c in repos.values())
    correct = sum(c.get('correct', 0) for c in repos.values())
    incorrect = sum(c.get('incorrect', 0) for c in repos.values())
    unknown = total - correct - incorrect
    print(f'{ds:<55} {n_repos:>7} {total:>14,} {correct:>9,} {incorrect:>11,} {unknown:>9,}')
    dataset_summary_rows.append({
        'Dataset': ds, '#Repos': n_repos, '#Trajectories': total,
        '#Correct': correct, '#Incorrect': incorrect, '#Unknown': unknown
    })

print('=' * 120)
total_all = len(all_data)
total_correct = sum(r['#Correct'] for r in dataset_summary_rows)
total_incorrect = sum(r['#Incorrect'] for r in dataset_summary_rows)
total_unknown = sum(r['#Unknown'] for r in dataset_summary_rows)
total_repos = sum(r['#Repos'] for r in dataset_summary_rows)
print(f'{"TOTAL":<55} {total_repos:>7} {total_all:>14,} {total_correct:>9,} {total_incorrect:>11,} {total_unknown:>9,}')

In [ ]:
# Visualize: Trajectories per dataset (bar chart)
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Bar chart: samples per dataset
ds_labels = [r['Dataset'][:40] for r in dataset_summary_rows]
ds_counts = [r['#Trajectories'] for r in dataset_summary_rows]
colors = plt.cm.tab20(np.linspace(0, 1, len(ds_labels)))

axes[0].barh(ds_labels, ds_counts, color=colors)
axes[0].set_xlabel('Number of Trajectories')
axes[0].set_title('Trajectories per Dataset (OPENSOURCE_v2)')
for i, v in enumerate(ds_counts):
    axes[0].text(v + max(ds_counts)*0.01, i, f'{v:,}', va='center', fontsize=8)

# Pie chart: correctness distribution
correctness_data = [total_correct, total_incorrect, total_unknown]
correctness_labels = ['Correct', 'Incorrect', 'Unknown']
correctness_colors = ['#2ecc71', '#e74c3c', '#95a5a6']
non_zero = [(l, d, c) for l, d, c in zip(correctness_labels, correctness_data, correctness_colors) if d > 0]
if non_zero:
    labels, data, cols = zip(*non_zero)
    axes[1].pie(data, labels=labels, colors=cols, autopct='%1.1f%%', startangle=90)
    axes[1].set_title('Correctness Distribution')

plt.tight_layout()
plt.show()

## 5. Issue Extraction & Deduplication Analysis (New in v2)

v2 introduces `issue_content_hash` for cross-dataset deduplication.

In [ ]:
# Analyze issue_content_hash coverage and uniqueness per dataset
hash_stats = defaultdict(lambda: {'total': 0, 'has_hash': 0, 'unique_hashes': set()})
all_hashes = []

for sample in full_dataset:
    ds = sample.get('source_dataset', 'unknown')
    hash_val = sample.get('issue_content_hash')
    hash_stats[ds]['total'] += 1
    if hash_val:
        hash_stats[ds]['has_hash'] += 1
        hash_stats[ds]['unique_hashes'].add(hash_val)
        all_hashes.append(hash_val)

print('=== Issue Content Hash Analysis ===')
print(f'{"Dataset":<55} {"Total":>8} {"Has Hash":>9} {"Unique":>8} {"Dup Ratio":>10}')
print('=' * 95)
for ds in sorted(hash_stats.keys()):
    s = hash_stats[ds]
    unique = len(s['unique_hashes'])
    dup_ratio = f'{1 - unique/s["has_hash"]:.2%}' if s['has_hash'] > 0 else 'N/A'
    print(f'{ds:<55} {s["total"]:>8,} {s["has_hash"]:>9,} {unique:>8,} {dup_ratio:>10}')

print('=' * 95)
total_hashes = len(all_hashes)
unique_global = len(set(all_hashes))
print(f'{"GLOBAL":<55} {len(full_dataset):>8,} {total_hashes:>9,} {unique_global:>8,} {1 - unique_global/total_hashes:.2%}' if total_hashes > 0 else '')
print(f'\nNote: High duplicate ratio within a dataset means many trajectories share the same issue.')
print(f'      This is expected for datasets with multiple attempts per issue (e.g., rejection sampling).')

In [ ]:
# Cross-dataset hash overlap analysis
print('=== Cross-Dataset Hash Overlap (Top pairs) ===')
print('Shows how many unique issues are shared between datasets.\n')

ds_names = sorted(hash_stats.keys())
overlaps = []
for i, ds1 in enumerate(ds_names):
    for ds2 in ds_names[i+1:]:
        overlap = len(hash_stats[ds1]['unique_hashes'] & hash_stats[ds2]['unique_hashes'])
        if overlap > 0:
            overlaps.append((ds1, ds2, overlap))

overlaps.sort(key=lambda x: -x[2])
print(f'{"Dataset 1":<40} {"Dataset 2":<40} {"Shared Issues":>13}')
print('-' * 95)
for ds1, ds2, overlap in overlaps[:20]:
    print(f'{ds1[:38]:<40} {ds2[:38]:<40} {overlap:>13,}')

## 6. Token Length Statistics

In [ ]:
# Compute token length statistics per dataset
dataset_token_stats = defaultdict(lambda: {'total': [], 'system': [], 'user': [], 'assistant': []})

for sample in all_data:
    ds = sample['source_dataset']
    msg_info = sample['msg_token_info']
    
    total_tokens = sum(t for _, t in msg_info)
    system_tokens = sum(t for r, t in msg_info if r == 'system')
    user_tokens = sum(t for r, t in msg_info if r == 'user')
    assistant_tokens = sum(t for r, t in msg_info if r == 'assistant')
    
    dataset_token_stats[ds]['total'].append(total_tokens)
    dataset_token_stats[ds]['system'].append(system_tokens)
    dataset_token_stats[ds]['user'].append(user_tokens)
    dataset_token_stats[ds]['assistant'].append(assistant_tokens)

# Build summary DataFrame
rows = []
for ds in sorted(dataset_token_stats.keys()):
    stats = dataset_token_stats[ds]
    row = {'Dataset': ds, 'N': len(stats['total'])}
    for category in ['total', 'system', 'user', 'assistant']:
        arr = np.array(stats[category])
        prefix = category.capitalize()
        row[f'{prefix}_Mean'] = int(arr.mean())
        row[f'{prefix}_Min'] = int(arr.min())
        row[f'{prefix}_Max'] = int(arr.max())
    rows.append(row)

df_dataset = pd.DataFrame(rows)

# Display as styled DataFrame
display_df = df_dataset.set_index('Dataset')
col_map = {
    'N': ('Count', 'N'),
    'Total_Mean': ('Total', 'Mean'), 'Total_Min': ('Total', 'Min'), 'Total_Max': ('Total', 'Max'),
    'System_Mean': ('System', 'Mean'), 'System_Min': ('System', 'Min'), 'System_Max': ('System', 'Max'),
    'User_Mean': ('User', 'Mean'), 'User_Min': ('User', 'Min'), 'User_Max': ('User', 'Max'),
    'Assistant_Mean': ('Assistant', 'Mean'), 'Assistant_Min': ('Assistant', 'Min'), 'Assistant_Max': ('Assistant', 'Max'),
}
display_df.columns = pd.MultiIndex.from_tuples([col_map[c] for c in display_df.columns])
display_df

In [ ]:
# Token length distribution visualization (global)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

all_totals = []
all_systems = []
all_users = []
all_assistants = []

for ds in sorted(dataset_token_stats.keys()):
    stats = dataset_token_stats[ds]
    all_totals.extend(stats['total'])
    all_systems.extend(stats['system'])
    all_users.extend(stats['user'])
    all_assistants.extend(stats['assistant'])

categories = [
    ('Total Tokens', all_totals),
    ('System Tokens', all_systems),
    ('User Tokens', all_users),
    ('Assistant Tokens', all_assistants),
]

for idx, (title, data) in enumerate(categories):
    ax = axes[idx // 2][idx % 2]
    arr = np.array(data)
    p99 = np.percentile(arr, 99)
    clipped = arr[arr <= p99]
    ax.hist(clipped, bins=50, edgecolor='black', alpha=0.7)
    ax.set_title(f'{title} Distribution (clipped at p99={int(p99):,})')
    ax.set_xlabel('Token Count')
    ax.set_ylabel('Frequency')
    ax.axvline(arr.mean(), color='red', linestyle='--', label=f'Mean={int(arr.mean()):,}')
    ax.axvline(np.median(arr), color='orange', linestyle='--', label=f'Median={int(np.median(arr)):,}')
    ax.legend()

plt.suptitle('SHINE_SWE_OPENSOURCE_v2 - Token Length Distributions (All Datasets)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Per-dataset token length comparison (box plot)
fig, ax = plt.subplots(figsize=(16, 10))

ds_names_sorted = sorted(dataset_token_stats.keys())
box_data = []
box_labels = []

for ds in ds_names_sorted:
    arr = np.array(dataset_token_stats[ds]['total'])
    # Clip at p95 for better visualization
    p95 = np.percentile(arr, 95)
    box_data.append(arr[arr <= p95])
    box_labels.append(ds[:35])

bp = ax.boxplot(box_data, vert=False, labels=box_labels, patch_artist=True)
colors = plt.cm.tab20(np.linspace(0, 1, len(box_data)))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_xlabel('Total Token Count (clipped at p95)')
ax.set_title('Token Length Distribution per Dataset (OPENSOURCE_v2)')
plt.tight_layout()
plt.show()

## 7. Top Repos by Trajectory Count

In [ ]:
# Top repos by number of trajectories
repo_counts = Counter(s['repo'] for s in all_data if s['repo'] != 'unknown')
top_repos = repo_counts.most_common(30)

total_unique_repos = len(repo_counts)
unknown_count = sum(1 for s in all_data if s['repo'] == 'unknown')

print(f'Total unique repos (excluding unknown): {total_unique_repos}')
print(f'Samples with repo="unknown": {unknown_count:,} ({unknown_count/len(all_data)*100:.1f}%)')
print(f'\nTop 30 repos by trajectory count:')
print(f'{"Repo":<50} {"Count":>8}')
print('-' * 60)
for repo, count in top_repos:
    print(f'{repo:<50} {count:>8,}')

# Visualize top repos
fig, ax = plt.subplots(figsize=(14, 10))
repos_labels = [r[0][:40] for r in top_repos[:20]]
repos_values = [r[1] for r in top_repos[:20]]
ax.barh(repos_labels[::-1], repos_values[::-1], color=plt.cm.viridis(np.linspace(0, 1, 20)))
ax.set_xlabel('Number of Trajectories')
ax.set_title('Top 20 Repos by Trajectory Count (OPENSOURCE_v2)')
plt.tight_layout()
plt.show()

## 8. New Datasets Deep Dive

Let's look at the 8 new datasets added in v2 in more detail.

In [ ]:
# New datasets in v2 (not present in v1)
v1_datasets = {
    'CoderForge-Preview', 'Kwai-Klear-SWE-smith-mini-swe-agent-plus-trajectories-66k',
    'Nemotron-SWE-v1', 'SWE-Gym-OpenHands-SFT-Trajectories',
    'SWE-Hero-openhands-trajectories', 'SWE-Lego-Real-Data',
    'SWE-Lego-Synthetic-Data', 'SWE-Zero-openhands-trajectories',
    'SWE-rebench-openhands-trajectories', 'SWE-smith-trajectories'
}

new_datasets = [ds for ds in sorted(dataset_repo_stats.keys()) if ds not in v1_datasets]
print(f'New datasets in v2 ({len(new_datasets)}):' )
print()

for ds in new_datasets:
    repos = dataset_repo_stats[ds]
    n_repos = len(repos)
    total = sum(sum(c.values()) for c in repos.values())
    correct = sum(c.get('correct', 0) for c in repos.values())
    incorrect = sum(c.get('incorrect', 0) for c in repos.values())
    unknown = total - correct - incorrect
    
    # Token stats
    ts = dataset_token_stats.get(ds, {'total': [0]})
    avg_tokens = int(np.mean(ts['total'])) if ts['total'] else 0
    
    print(f'  📦 {ds}')
    print(f'     Samples: {total:,} | Repos: {n_repos} | Avg Tokens: {avg_tokens:,}')
    print(f'     Correct: {correct:,} | Incorrect: {incorrect:,} | Unknown: {unknown:,}')
    
    # Hash info
    hs = hash_stats.get(ds, {'has_hash': 0, 'unique_hashes': set()})
    print(f'     Unique Issues (by hash): {len(hs["unique_hashes"]):,}')
    print()

## 9. Comparison with v1

Quick summary of key differences from SHINE_SWE_OPENSOURCE (v1):
- **v1**: 10 datasets, ~1,081,835 samples (JSONL format)
- **v2**: 18 datasets, significantly more samples (Arrow format)
- **CoderForge-Preview**: Deduplicated from 413K to 258K (by trajectory_id)
- **New data sources**: nvidia, GAIR, Kimi K2, AweAI-Team, ernie-research
- **All datasets**: Now have `issue_content_hash` for cross-dataset deduplication

In [ ]:
# Summary comparison
print('=== v1 vs v2 Comparison ===')
print(f'{"Metric":<40} {"v1":>15} {"v2":>15}')
print('=' * 72)
print(f'{"Number of datasets":<40} {10:>15} {len(dataset_summary_rows):>15}')
print(f'{"Total samples":<40} {"1,081,835":>15} {total_all:>15,}')
print(f'{"Data format":<40} {"JSONL":>15} {"Arrow":>15}')
print(f'{"Has issue_content_hash":<40} {"No":>15} {"Yes":>15}')
print(f'{"CoderForge-Preview samples":<40} {"413,278":>15} {"258,134 (dedup)":>15}')
print(f'{"New datasets added":<40} {"-":>15} {len(new_datasets):>15}')
print()
print('New datasets added in v2:')
for ds in new_datasets:
    total = sum(sum(c.values()) for c in dataset_repo_stats[ds].values())
    print(f'  + {ds}: {total:,} samples')